# Phan tich Du lieu - GestuRhythm v2
Sinh tat ca anh phan tich du lieu cho bao cao hoc thuat.
Moi anh luu vao /kaggle/working/ de download.

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch
from scipy import stats
import os

# ── Academic style ──────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':       'serif',
    'font.size':         11,
    'axes.titlesize':    12,
    'axes.labelsize':    11,
    'xtick.labelsize':   10,
    'ytick.labelsize':   10,
    'legend.fontsize':   10,
    'figure.dpi':        150,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         True,
    'grid.alpha':        0.3,
    'grid.linestyle':    '--',
})
SAVE_DIR = '/kaggle/working/'

# Load data
for p in ['/kaggle/input/datasets/quangleuq/v3-data/',
          '/kaggle/input/gesturhythm-v2/',
          '/kaggle/input/v3-data/']:
    if os.path.exists(p + 'sequences.npy'):
        DATA_DIR = p; break

X  = np.load(DATA_DIR + 'sequences.npy')   # (N, 30, 225)
y  = np.load(DATA_DIR + 'emotions.npy')    # (N, 2)
m  = np.load(DATA_DIR + 'mode_ids.npy')    # (N,)
N, T, F = X.shape
print(f'sequences: {X.shape} | emotions: {y.shape} | modes: {m.shape}')

MODE_NAMES  = {0:'HAPPY HIGH', 1:'HAPPY LOW', 2:'SAD HIGH', 3:'SAD LOW', 4:'NEUTRAL', 5:'NONE'}
MODE_COLORS = {0:'#27ae60', 1:'#2980b9', 2:'#c0392b', 3:'#8e44ad', 4:'#e67e22', 5:'#7f8c8d'}
MODE_MARKERS= {0:'o', 1:'s', 2:'^', 3:'D', 4:'P', 5:'X'}
N_MODES = 6
print('Setup OK')


## Hinh 1: Khong gian cam xuc Valence-Arousal (Figure 1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Hinh 1: Phan bo du lieu trong khong gian Valence-Arousal', fontsize=13, fontweight='bold', y=1.01)

# (a) Scatter toan bo
ax = axes[0]
for mid in range(N_MODES):
    idx = (m == mid)
    ax.scatter(y[idx, 0], y[idx, 1],
               c=MODE_COLORS[mid], marker=MODE_MARKERS[mid],
               label=MODE_NAMES[mid], alpha=0.6, s=25, edgecolors='none')
ax.axhline(0, color='black', linewidth=0.8, linestyle=':')
ax.axvline(0, color='black', linewidth=0.8, linestyle=':')
ax.text( 0.65,  0.85, 'Vui + Nang luong', fontsize=8, color='gray', transform=ax.transAxes)
ax.text( 0.65,  0.08, 'Vui + Thu gian',   fontsize=8, color='gray', transform=ax.transAxes)
ax.text( 0.02,  0.85, 'Buon + Cang thang',fontsize=8, color='gray', transform=ax.transAxes)
ax.text( 0.02,  0.08, 'Buon + Cham',      fontsize=8, color='gray', transform=ax.transAxes)
ax.set_xlabel('Valence (buon ← 0 → vui)')
ax.set_ylabel('Arousal (thu gian ← 0 → nang luong)')
ax.set_title('(a) Phan bo toan bo dataset')
ax.legend(loc='center', bbox_to_anchor=(0.5,-0.18), ncol=3, frameon=True)
ax.set_xlim(-1.3, 1.3); ax.set_ylim(-1.3, 1.3)

# (b) Trung tam va do lech chuan moi mode
ax2 = axes[1]
for mid in range(N_MODES):
    idx = (m == mid)
    mu_v, mu_a = y[idx,0].mean(), y[idx,1].mean()
    sd_v, sd_a = y[idx,0].std(),  y[idx,1].std()
    ax2.errorbar(mu_v, mu_a, xerr=sd_v, yerr=sd_a,
                 fmt=MODE_MARKERS[mid], color=MODE_COLORS[mid],
                 capsize=4, markersize=10, label=MODE_NAMES[mid], linewidth=1.5)
ax2.axhline(0, color='black', linewidth=0.8, linestyle=':')
ax2.axvline(0, color='black', linewidth=0.8, linestyle=':')
ax2.set_xlabel('Valence trung binh (+/- std)')
ax2.set_ylabel('Arousal trung binh (+/- std)')
ax2.set_title('(b) Trung tam va do lech chuan moi che do')
ax2.legend(loc='center', bbox_to_anchor=(0.5,-0.18), ncol=3, frameon=True)
ax2.set_xlim(-1.5, 1.5); ax2.set_ylim(-1.5, 1.5)

plt.tight_layout()
plt.savefig(SAVE_DIR + 'fig01_emotion_space.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved fig01_emotion_space.png')


## Hinh 2: Phan bo so luong mau va chat luong (Figure 2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Hinh 2: Thong ke va chat luong dataset', fontsize=13, fontweight='bold')

# (a) So luong mau moi mode
ax = axes[0]
counts = [np.sum(m == i) for i in range(N_MODES)]
bars = ax.bar([MODE_NAMES[i] for i in range(N_MODES)], counts,
              color=[MODE_COLORS[i] for i in range(N_MODES)], edgecolor='black', linewidth=0.5)
for bar, c in zip(bars, counts):
    ax.text(bar.get_x()+bar.get_width()/2, c+2, str(c), ha='center', fontsize=9)
ax.set_title('(a) So luong mau moi che do')
ax.set_ylabel('So mau'); ax.set_xlabel('Che do cam xuc')
ax.tick_params(axis='x', rotation=30)
ax.axhline(np.mean(counts), color='red', linestyle='--', linewidth=1, label=f'Trung binh={np.mean(counts):.0f}')
ax.legend()

# (b) Ti le frames co landmarks
ax2 = axes[1]
coverage = []
for i in range(N_MODES):
    idx = np.where(m == i)[0]
    if len(idx) == 0: coverage.append(0); continue
    seqs = X[idx]  # (n, 30, 225)
    nonzero = (np.abs(seqs).sum(axis=2) > 0).mean()
    coverage.append(nonzero * 100)
bars2 = ax2.bar([MODE_NAMES[i] for i in range(N_MODES)], coverage,
                color=[MODE_COLORS[i] for i in range(N_MODES)], edgecolor='black', linewidth=0.5)
for bar, c in zip(bars2, coverage):
    ax2.text(bar.get_x()+bar.get_width()/2, c+0.5, f'{c:.1f}%', ha='center', fontsize=8)
ax2.set_title('(b) Ti le frame co landmarks hop le')
ax2.set_ylabel('Ti le (%)'); ax2.set_xlabel('Che do cam xuc')
ax2.tick_params(axis='x', rotation=30); ax2.set_ylim(0, 110)

# (c) Phan bo gia tri landmarks (boxplot)
ax3 = axes[2]
sample_features = X[:, :, :9].reshape(-1, 9)  # 9 features dau
sample_features = sample_features[np.abs(sample_features).sum(1) > 0]
ax3.boxplot(sample_features[:2000], patch_artist=True,
            boxprops=dict(facecolor='lightblue', alpha=0.7),
            medianprops=dict(color='red', linewidth=2),
            flierprops=dict(marker='.', markersize=2, alpha=0.3))
ax3.set_title('(c) Phan bo gia tri landmarks (9 features dau)')
ax3.set_xlabel('Feature index'); ax3.set_ylabel('Gia tri normalized')
ax3.axhline(0, color='black', linewidth=0.5)

plt.tight_layout()
plt.savefig(SAVE_DIR + 'fig02_dataset_stats.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved fig02_dataset_stats.png')


## Hinh 3: Phan tich chuoi thoi gian (Figure 3)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('Hinh 3: Phan tich chuoi cu chi theo thoi gian (30 frames ~ 1 giay)',
             fontsize=13, fontweight='bold')

# Tinh mean landmark magnitude moi frame cho tung mode
for plot_idx, mid in enumerate(range(N_MODES)):
    ax = axes[plot_idx // 3][plot_idx % 3]
    idx = np.where(m == mid)[0]
    if len(idx) == 0:
        ax.text(0.5, 0.5, 'Khong co data', ha='center', transform=ax.transAxes)
        continue
    seqs = X[idx[:min(20, len(idx))]]  # lay toi da 20 sample
    # Mean magnitude cua tay phai (63 features dau)
    rh_mag = np.linalg.norm(seqs[:, :, :63].reshape(len(seqs), T, 21, 3), axis=3).mean(axis=2)
    frames = np.arange(T)
    mu  = rh_mag.mean(axis=0)
    std = rh_mag.std(axis=0)
    ax.plot(frames, mu, color=MODE_COLORS[mid], linewidth=2, label='Trung binh')
    ax.fill_between(frames, mu-std, mu+std, alpha=0.2, color=MODE_COLORS[mid], label='±1 std')
    for s in seqs[:5]:
        mag = np.linalg.norm(s[:, :63].reshape(T, 21, 3), axis=2).mean(axis=1)
        ax.plot(frames, mag, color=MODE_COLORS[mid], alpha=0.15, linewidth=0.8)
    ax.set_title(f'({chr(97+plot_idx)}) {MODE_NAMES[mid]}')
    ax.set_xlabel('Frame (0-29)'); ax.set_ylabel('Magnitude landmarks')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(SAVE_DIR + 'fig03_time_series.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved fig03_time_series.png')


## Hinh 4: So sanh dac trung giua cac mode (Figure 4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Hinh 4: So sanh dac trung dong hoc giua 6 che do cam xuc',
             fontsize=13, fontweight='bold')

# (a) Toc do thay doi trung binh (velocity)
ax = axes[0]
velocities = {}
for mid in range(N_MODES):
    idx = np.where(m == mid)[0]
    if len(idx) == 0: velocities[mid] = np.zeros(T-1); continue
    seqs = X[idx, :, :63]  # tay phai
    vel  = np.linalg.norm(np.diff(seqs, axis=1), axis=2).mean(axis=0)  # (T-1,)
    velocities[mid] = vel
    ax.plot(np.arange(T-1), vel, color=MODE_COLORS[mid],
            linewidth=2, marker=MODE_MARKERS[mid], markersize=4,
            markevery=5, label=MODE_NAMES[mid])
ax.set_title('(a) Toc do trung binh landmark tay phai')
ax.set_xlabel('Frame'); ax.set_ylabel('Toc do (||delta landmark||)')
ax.legend(ncol=2, fontsize=9)

# (b) Boxplot toc do trung binh moi mode
ax2 = axes[1]
data_box = []
labels_box = []
for mid in range(N_MODES):
    idx = np.where(m == mid)[0]
    if len(idx) == 0: continue
    seqs = X[idx, :, :63]
    vel_per_sample = np.linalg.norm(np.diff(seqs, axis=1), axis=2).mean(axis=(1,2))
    data_box.append(vel_per_sample)
    labels_box.append(MODE_NAMES[mid])
bp = ax2.boxplot(data_box, labels=labels_box, patch_artist=True,
                 medianprops=dict(color='black', linewidth=2))
for patch, mid in zip(bp['boxes'], range(N_MODES)):
    patch.set_facecolor(MODE_COLORS[mid])
    patch.set_alpha(0.7)
ax2.set_title('(b) Phan bo toc do trung binh moi che do')
ax2.set_xlabel('Che do cam xuc'); ax2.set_ylabel('Toc do trung binh')
ax2.tick_params(axis='x', rotation=25)

plt.tight_layout()
plt.savefig(SAVE_DIR + 'fig04_motion_features.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved fig04_motion_features.png')


## Hinh 5: Heatmap tuong quan va PCA (Figure 5)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Hinh 5: Phan tich tuong quan va phan tich thanh phan chinh (PCA)',
             fontsize=13, fontweight='bold')

# (a) Heatmap tuong quan 9 features dau
ax = axes[0]
feat_names = [f'RH_lm{i//3}_{["x","y","z"][i%3]}' for i in range(9)]
sample = X[:, 15, :9]  # frame giua
sample = sample[np.abs(sample).sum(1) > 0]
corr = np.corrcoef(sample.T)
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(9)); ax.set_xticklabels(feat_names, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(9)); ax.set_yticklabels(feat_names, fontsize=8)
for i in range(9):
    for j in range(9):
        ax.text(j, i, f'{corr[i,j]:.2f}', ha='center', va='center', fontsize=7,
                color='white' if abs(corr[i,j]) > 0.5 else 'black')
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title('(a) Ma tran tuong quan landmarks (9 features)')

# (b) PCA 2D
ax2 = axes[1]
X_flat = X.reshape(N, T*F)
X_flat = X_flat[np.abs(X_flat).sum(1) > 0]
m_filt = m[np.abs(X.reshape(N, T*F)).sum(1) > 0] if len(X_flat) == N else m[:len(X_flat)]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_flat[:500])
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
for mid in range(N_MODES):
    idx_pca = [i for i, mi in enumerate(m_filt[:500]) if mi == mid]
    if idx_pca:
        ax2.scatter(X_pca[idx_pca, 0], X_pca[idx_pca, 1],
                    c=MODE_COLORS[mid], marker=MODE_MARKERS[mid],
                    label=MODE_NAMES[mid], alpha=0.5, s=20, edgecolors='none')
ax2.set_title(f'(b) PCA 2D (var explained: {pca.explained_variance_ratio_.sum()*100:.1f}%)')
ax2.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax2.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax2.legend(ncol=2, fontsize=9)

plt.tight_layout()
plt.savefig(SAVE_DIR + 'fig05_correlation_pca.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved fig05_correlation_pca.png')


## In danh sach tat ca anh da luu

In [ ]:
import glob
saved = sorted(glob.glob(SAVE_DIR + 'fig*.png'))
print(f'Da luu {len(saved)} anh:')
for f in saved:
    size_kb = os.path.getsize(f) // 1024
    print(f'  {os.path.basename(f):40s} {size_kb:4d} KB')
